# Assignment 2: Spark Pipeline
Jia Hao Huang Xia


## Dataset: NYC Yellow Taxi Trip Data Full Year
You will work with real trip records from the New York City Taxi and Limousine Commission
(TLC). The data is publicly available and already in Parquet format.

Download 12 months of 2024 yellow taxi data (~660 MB in Parquet, ~41 million rows).
- https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet

to
- https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-12.parquet

In [1]:
import urllib.request
import os
import glob
import warnings
import time

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, min, max, hour, month, dayofweek, unix_timestamp, avg, sum

warnings.filterwarnings('ignore')

In [2]:
for month in range(1, 13):
    url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-{month:02d}.parquet"
    dest = f"/app/data/yellow_tripdata_2024-{month:02d}.parquet"
    print(f"Downloading 2024-{month:02d}...")
    urllib.request.urlretrieve(url, dest)

In [2]:
spark = SparkSession.builder \
    .appName("Assignment2") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/10 21:10:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
#spark.stop()

In [4]:
df_jan = spark.read.parquet("/app/data/yellow_tripdata_2024-01.parquet")

df_jan.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [5]:
df_jan

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double]

# 1. Spark RDDs

In [6]:
# 1.1
rdd_jan = df_jan.rdd
first_3 = rdd_jan.take(3)
for row in first_3:
    print(row)
    print(type(row))
    print()

[Stage 2:=============================>                             (2 + 2) / 4]

Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2024, 1, 1, 0, 57, 55), tpep_dropoff_datetime=datetime.datetime(2024, 1, 1, 1, 17, 43), passenger_count=1, trip_distance=1.72, RatecodeID=1, store_and_fwd_flag='N', PULocationID=186, DOLocationID=79, payment_type=2, fare_amount=17.7, extra=1.0, mta_tax=0.5, tip_amount=0.0, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=22.7, congestion_surcharge=2.5, Airport_fee=0.0)
<class 'pyspark.sql.types.Row'>

Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2024, 1, 1, 0, 3), tpep_dropoff_datetime=datetime.datetime(2024, 1, 1, 0, 9, 36), passenger_count=1, trip_distance=1.8, RatecodeID=1, store_and_fwd_flag='N', PULocationID=140, DOLocationID=236, payment_type=1, fare_amount=10.0, extra=3.5, mta_tax=0.5, tip_amount=3.75, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=18.75, congestion_surcharge=2.5, Airport_fee=0.0)
<class 'pyspark.sql.types.Row'>

Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2024, 1, 1,

In [ ]:
# 1.2
print("Total_trip: ", rdd_jan.count())

[Stage 13:===========================================>              (6 + 2) / 8]

Total_trip:  2964624


In [12]:
passenger = rdd_jan.filter(lambda row: row.passenger_count is not None and row.passenger_count > 4)
print("Trips with passenger_count > 4:", passenger.count())

[Stage 14:===========================================>              (6 + 2) / 8]

Trips with passenger_count > 4: 55919


In [13]:
trip_distance = rdd_jan.map(lambda row: (row.trip_distance, 1)).reduce(lambda a, b: (a[0] + b[0], a[1] + b[1]))
avg_trip_distance = trip_distance[0] / trip_distance[1]
print("Average trip distance:", avg_trip_distance)

[Stage 15:===========================================>              (6 + 2) / 8]

Average trip distance: 3.6521691789583146


In [ ]:
# 1.3
revenue_per_location = rdd_jan.filter(lambda row: row.PULocationID is not None and row.total_amount is not None)\
    .map(lambda row: (row.PULocationID, row.total_amount)).reduceByKey(lambda a, b: a + b)

top_5 = (revenue_per_location.sortBy(lambda x: x[1], ascending=False).take(5))

for location in top_5:
    print(location)

(132, 11121928.529997837)
(138, 5820969.9599999)
(161, 3369045.009999964)
(230, 2793051.6799999666)
(237, 2776195.3199999486)


# 2. Dataframes: Cleaning and Transforming

In [ ]:
# 2.1
df = spark.read.parquet("/app/data/yellow_tripdata_2024-*.parquet")
df.printSchema()

26/05/10 21:11:07 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /app/data/yellow_tripdata_2024-*.parquet.
java.io.FileNotFoundException: File /app/data/yellow_tripdata_2024-*.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [ ]:
# 2.2
total_rows = df.count()
print("Total rows:", df.count())
print("Total columns:", len(df.columns))

Total rows: 41169720
Total columns: 19


In [17]:
null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
null_counts.show(vertical=True, truncate=False)

[Stage 30:======================================>                  (8 + 4) / 12]

-RECORD 0------------------------
 VendorID              | 0       
 tpep_pickup_datetime  | 0       
 tpep_dropoff_datetime | 0       
 passenger_count       | 4091232 
 trip_distance         | 0       
 RatecodeID            | 4091232 
 store_and_fwd_flag    | 4091232 
 PULocationID          | 0       
 DOLocationID          | 0       
 payment_type          | 0       
 fare_amount           | 0       
 extra                 | 0       
 mta_tax               | 0       
 tip_amount            | 0       
 tolls_amount          | 0       
 improvement_surcharge | 0       
 total_amount          | 0       
 congestion_surcharge  | 4091232 
 Airport_fee           | 4091232 



In [18]:
df.select(
    min("tpep_pickup_datetime").alias("min_pickup"),
    max("tpep_pickup_datetime").alias("max_pickup")
).show(truncate=False)

[Stage 33:======================================>                  (8 + 4) / 12]

+-------------------+-------------------+
|min_pickup         |max_pickup         |
+-------------------+-------------------+
|2002-12-31 16:46:07|2026-06-26 23:53:12|
+-------------------+-------------------+



In [19]:
df.filter(
    (col("tpep_pickup_datetime") < "2024-01-01") |
    (col("tpep_pickup_datetime") >= "2025-01-01")
).select("tpep_pickup_datetime").show()

+--------------------+
|tpep_pickup_datetime|
+--------------------+
| 2009-01-01 00:35:59|
| 2002-12-31 16:46:07|
| 2009-01-01 00:13:29|
| 2009-01-01 00:01:41|
| 2008-12-31 23:04:30|
| 2002-12-31 23:10:22|
| 2002-12-31 23:19:30|
| 2009-01-01 05:36:19|
| 2025-02-09 03:46:10|
| 2025-03-02 11:38:31|
| 2025-03-23 20:42:06|
| 2008-12-31 23:58:01|
| 2025-01-06 13:52:50|
| 2025-01-16 10:16:49|
| 2009-01-01 00:06:17|
| 2008-12-31 23:03:59|
| 2008-12-31 23:05:26|
| 2009-01-01 14:02:23|
| 2008-12-31 23:03:46|
| 2009-01-01 00:38:49|
+--------------------+
only showing top 20 rows


In [20]:
df.groupBy("payment_type").count().orderBy(col("count").desc()).show()

[Stage 38:======================================>                  (8 + 4) / 12]

+------------+--------+
|payment_type|   count|
+------------+--------+
|           1|30452159|
|           2| 5540088|
|           0| 4091232|
|           4|  794494|
|           3|  291743|
|           5|       4|
+------------+--------+



In [5]:
# 2.3
df_clean = df.filter((col("trip_distance") > 0) & (col("total_amount") > 0))
step1 = df_clean.count()
removed_step1 = df.count() - step1
print("Initial rows:", df.count())
print("Removed invalid trips:", removed_step1)

Initial rows: 41169720
Removed invalid trips: 1336887


In [6]:
df_clean = df_clean.filter(col("passenger_count").isNotNull() & (col("passenger_count") > 0))
step2 = df_clean.count()
removed_step2 = step1 - step2
print("Removed invalid passenger counts:", removed_step2)

[Stage 16:======================================>                  (8 + 4) / 12]

Removed invalid passenger counts: 4200008


In [7]:
df_clean = df_clean.filter((col("trip_distance") <= 200) & (col("total_amount") <= 5000))
step3 = df_clean.count()
removed_step3 = step2 - step3
print("Removed outliers:", removed_step3)

[Stage 19:======================================>                  (8 + 4) / 12]

Removed outliers: 268


In [8]:
df_clean = df_clean.filter((col("tpep_pickup_datetime") >= "2024-01-01") & (col("tpep_pickup_datetime") < "2025-01-01"))
step4 = df_clean.count()
removed_step4 = step3 - step4
print("Removed time anomalies:", removed_step4)

[Stage 22:======================================>                  (8 + 4) / 12]

Removed time anomalies: 55


In [9]:
print("Initial rows:", df.count())
print("Invalid trips removed:", removed_step1)
print("Invalid passenger count removed:", removed_step2)
print("Outliers removed:", removed_step3)
print("Time anomalies removed:", removed_step4)
print("Final rows:", step4)
total_removed = df.count() - step4
percentage_removed = (total_removed / df.count()) * 100
print("Total removed:", total_removed)
print(f"Total removed percentage: {percentage_removed:.2f}%")

Initial rows: 41169720
Invalid trips removed: 1336887
Invalid passenger count removed: 4200008
Outliers removed: 268
Time anomalies removed: 55
Final rows: 35632502
Total removed: 5537218
Total removed percentage: 13.45%


In [10]:
# 2.4
df_clean = df_clean.withColumn("trip_duration_minutes", (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60)
df_clean = df_clean.withColumn("hour_of_day",hour("tpep_pickup_datetime"))
df_clean = df_clean.withColumn("day_of_week", when(dayofweek("tpep_pickup_datetime") == 1, 7).otherwise(dayofweek("tpep_pickup_datetime") - 1))
df_clean = df_clean.withColumn("month", month("tpep_pickup_datetime"))
df_clean = df_clean.withColumn("tip_percentage", when(col("fare_amount") > 0, (col("tip_amount") / col("fare_amount")) * 100).otherwise(None))
df_clean = df_clean.withColumn("speed_mph", when(col("trip_duration_minutes") > 0, col("trip_distance") / (col("trip_duration_minutes") / 60)).otherwise(None))
df_clean = df_clean.filter((col("speed_mph").isNull()) | (col("speed_mph") <= 100))


In [31]:
df_clean.select(
    "trip_duration_minutes",
    "hour_of_day",
    "day_of_week",
    "month",
    "tip_percentage",
    "speed_mph"
).show(5)

+---------------------+-----------+-----------+-----+------------------+------------------+
|trip_duration_minutes|hour_of_day|day_of_week|month|    tip_percentage|         speed_mph|
+---------------------+-----------+-----------+-----+------------------+------------------+
|                 17.7|          0|          2|   10| 8.152173913043478| 10.16949152542373|
|   13.083333333333334|          0|          2|   10| 26.76056338028169|10.089171974522294|
|                  9.1|          0|          2|   10| 27.40740740740741|17.802197802197803|
|                10.85|          0|          2|   10|14.084507042253522|17.142857142857146|
|    4.666666666666667|          0|          2|   10|33.888888888888886|12.471428571428572|
+---------------------+-----------+-----------+-----+------------------+------------------+
only showing top 5 rows


In [32]:
# 2.5
hourly = df_clean.groupBy("hour_of_day").agg(
    avg("trip_distance").alias("avg_trip_distance"),
    avg("fare_amount").alias("avg_fare_amount")
).orderBy("hour_of_day")
hourly.show(24, truncate=False)

[Stage 69:==============================================>         (10 + 2) / 12]

+-----------+------------------+------------------+
|hour_of_day|avg_trip_distance |avg_fare_amount   |
+-----------+------------------+------------------+
|0          |3.9940352209452854|20.30939805135777 |
|1          |3.451589763554973 |18.065711786363174|
|2          |3.093273722564794 |16.579505236369506|
|3          |3.36976697139736  |17.58139394102555 |
|4          |4.964229166791558 |23.816265099242834|
|5          |6.338921588625854 |28.514970320356465|
|6          |4.942863689279742 |23.142339296536758|
|7          |3.743502452714594 |19.53600729558587 |
|8          |3.224630307714529 |18.576205212677593|
|9          |3.1320892408323777|18.649960359949574|
|10         |3.1565001066716376|19.039094384865432|
|11         |3.130636535000449 |19.304061133703566|
|12         |3.221441404134355 |19.73083647909722 |
|13         |3.412610298999755 |20.460779123627834|
|14         |3.56983635504181  |21.25179492323656 |
|15         |3.5365677995660953|21.236029609032595|
|16         

In [33]:
hourly.orderBy(col("avg_trip_distance").desc()).show(5)

[Stage 72:======================================>                  (8 + 4) / 12]

+-----------+------------------+------------------+
|hour_of_day| avg_trip_distance|   avg_fare_amount|
+-----------+------------------+------------------+
|          5| 6.338921588625854|28.514970320356465|
|          4| 4.964229166791558|23.816265099242834|
|          6| 4.942863689279742|23.142339296536758|
|         23|   4.0789334949131| 20.96333551394519|
|          0|3.9940352209452854| 20.30939805135777|
+-----------+------------------+------------------+
only showing top 5 rows


In [34]:
hourly.orderBy(col("avg_fare_amount").desc()).show(5)

[Stage 75:======================================>                  (8 + 4) / 12]

+-----------+-----------------+------------------+
|hour_of_day|avg_trip_distance|   avg_fare_amount|
+-----------+-----------------+------------------+
|          5|6.338921588625854|28.514970320356465|
|          4|4.964229166791558|23.816265099242834|
|          6|4.942863689279742|23.142339296536758|
|         16|3.558670158640665| 21.29922776964351|
|         14| 3.56983635504181| 21.25179492323656|
+-----------+-----------------+------------------+
only showing top 5 rows


In [35]:
df_clean.groupBy("month").agg(sum("total_amount").alias("total_revenue")).orderBy("month").show()

[Stage 78:==========================================>              (9 + 3) / 12]

+-----+-------------------+
|month|      total_revenue|
+-----+-------------------+
|    1|7.461821098993786E7|
|    2|7.444384533994398E7|
|    3|8.601527200986908E7|
|    4|8.579874491987401E7|
|    5|9.370999945984167E7|
|    6|8.745999504986383E7|
|    7|7.899976238991883E7|
|    8|7.722124146991742E7|
|    9|9.131358298986937E7|
|   10|9.870875900983047E7|
|   11|9.077769155985183E7|
|   12|9.445349867983595E7|
+-----+-------------------+



In [36]:
trips_per_day = df_clean.groupBy("day_of_week").agg(count("*").alias("trip_count")).orderBy("day_of_week")
trips_per_day.show()

[Stage 81:==========================================>              (9 + 3) / 12]

+-----------+----------+
|day_of_week|trip_count|
+-----------+----------+
|          1|   4547041|
|          2|   5200650|
|          3|   5369091|
|          4|   5584259|
|          5|   5282992|
|          6|   5236548|
|          7|   4402983|
+-----------+----------+



In [37]:
trips_per_day.orderBy(col("trip_count").desc()).show()

+-----------+----------+
|day_of_week|trip_count|
+-----------+----------+
|          4|   5584259|
|          3|   5369091|
|          5|   5282992|
|          6|   5236548|
|          2|   5200650|
|          1|   4547041|
|          7|   4402983|
+-----------+----------+



In [38]:
df_clean.groupBy("payment_type").agg(avg("tip_percentage").alias("avg_tip_percentage")).orderBy(col("avg_tip_percentage").desc()).show()

[Stage 87:======================================>                  (8 + 4) / 12]

+------------+--------------------+
|payment_type|  avg_tip_percentage|
+------------+--------------------+
|           1|  25.097827415209125|
|           4| 0.06306863370604723|
|           3| 0.06155066243447759|
|           2|0.001491349102521...|
+------------+--------------------+



In [11]:
# 2.6
# Overwrite to ensure rerunning the notebook does not fail
df_clean.coalesce(1).write.mode("overwrite").parquet("/app/data/nyc_taxi_2024_clean.parquet")
print(os.listdir("/app/data/nyc_taxi_2024_clean.parquet"))

['.part-00000-da024de3-0cad-4af5-9ec7-7bc4ced6b3bc-c000.snappy.parquet.crc', '._SUCCESS.crc', 'part-00000-da024de3-0cad-4af5-9ec7-7bc4ced6b3bc-c000.snappy.parquet', '_SUCCESS']


# 3. Spark SQL & Windows functions

In [12]:
df_clean.createOrReplaceTempView("trips")

In [42]:
# 3.1
top_locations = spark.sql("""
SELECT
    PULocationID,
    COUNT(*) AS trip_count
FROM trips
GROUP BY PULocationID
ORDER BY trip_count DESC
LIMIT 10
""")

top_locations.show()

[Stage 91:==========================================>              (9 + 3) / 12]

+------------+----------+
|PULocationID|trip_count|
+------------+----------+
|         132|   1856382|
|         237|   1766762|
|         161|   1737647|
|         236|   1557671|
|         162|   1309910|
|         186|   1269313|
|         230|   1250688|
|         138|   1245356|
|         142|   1182683|
|         163|   1063249|
+------------+----------+



In [43]:
payment_stats = spark.sql("""
SELECT
    payment_type,
    AVG(fare_amount) AS avg_fare,
    AVG(tip_amount) AS avg_tip,
    AVG(total_amount) AS avg_total
FROM trips
GROUP BY payment_type
ORDER BY payment_type
""")

payment_stats.show()

[Stage 94:==========================================>              (9 + 3) / 12]

+------------+------------------+--------------------+------------------+
|payment_type|          avg_fare|             avg_tip|         avg_total|
+------------+------------------+--------------------+------------------+
|           1|19.746730828711478|    4.35795764517939|29.753365123079842|
|           2|19.636604169403547|2.381840884178569...| 24.95892488719613|
|           3| 19.94209428067393|0.006390845307513097|25.267221676071134|
|           4|22.256116792579085| 0.01376143563621591|27.913546720416065|
+------------+------------------+--------------------+------------------+



In [44]:
distances = spark.sql("""
WITH overall_avg AS (
    SELECT AVG(trip_distance) AS overall_distance
    FROM trips
),

location_avg AS (
    SELECT
        PULocationID,
        AVG(trip_distance) AS avg_distance
    FROM trips
    GROUP BY PULocationID
)

SELECT
    l.PULocationID,
    l.avg_distance,
    o.overall_distance
FROM location_avg l
CROSS JOIN overall_avg o
WHERE l.avg_distance > 2 * o.overall_distance
ORDER BY l.avg_distance DESC
""")

distances.show()

[Stage 98:==========================================>              (9 + 3) / 12]

+------------+------------------+------------------+
|PULocationID|      avg_distance|  overall_distance|
+------------+------------------+------------------+
|          44|            22.875|3.4374351914925456|
|         156|18.413333333333334|3.4374351914925456|
|         132| 15.96060854931797|3.4374351914925456|
|          86|15.776257253384912|3.4374351914925456|
|          23|15.224499999999997|3.4374351914925456|
|           2| 14.90076923076923|3.4374351914925456|
|         117|14.449293833964864|3.4374351914925456|
|         115|13.928999999999998|3.4374351914925456|
|         201|13.908612862547287|3.4374351914925456|
|         118|13.837222222222222|3.4374351914925456|
|          46| 13.56572192513369|3.4374351914925456|
|         219|13.381684287276707|3.4374351914925456|
|         139|13.001226190476189|3.4374351914925456|
|          10|12.940180454845086|3.4374351914925456|
|          38| 12.79779322853688|3.4374351914925456|
|         259|12.706564569536424|3.43743519149

In [45]:
# 3.2
time_of_day_trips = spark.sql("""
SELECT
    CASE
        WHEN hour_of_day BETWEEN 6 AND 11 THEN 'morning'
        WHEN hour_of_day BETWEEN 12 AND 17 THEN 'afternoon'
        WHEN hour_of_day BETWEEN 18 AND 22 THEN 'evening'
        ELSE 'night'
    END AS time_of_day,
    AVG(trip_distance) AS avg_trip_distance,
    AVG(total_amount) AS avg_total_amount,
    AVG(tip_percentage) AS avg_tip_percentage
FROM trips
GROUP BY
    CASE
        WHEN hour_of_day BETWEEN 6 AND 11 THEN 'morning'
        WHEN hour_of_day BETWEEN 12 AND 17 THEN 'afternoon'
        WHEN hour_of_day BETWEEN 18 AND 22 THEN 'evening'
        ELSE 'night'
    END
ORDER BY avg_total_amount DESC
""")

time_of_day_trips.show()

[Stage 103:=====================================>                  (8 + 4) / 12]

+-----------+------------------+------------------+------------------+
|time_of_day| avg_trip_distance|  avg_total_amount|avg_tip_percentage|
+-----------+------------------+------------------+------------------+
|  afternoon|3.4164588028984992| 30.00134694717678| 20.34933824213464|
|      night|3.9625570681088558|29.492232544607976|21.451774411315228|
|    evening|3.3407432265304293|28.717754586344608|22.551773768030156|
|    morning|3.3350354849338553|27.476839481322337|19.922032409207173|
+-----------+------------------+------------------+------------------+



In [46]:
# 3.3
monthly_ranking = spark.sql("""
WITH monthly_revenue AS (
    SELECT
        month,
        PULocationID,
        SUM(total_amount) AS revenue
    FROM trips
    GROUP BY month, PULocationID
),

locations AS (
    SELECT
        month,
        PULocationID,
        revenue,
        ROW_NUMBER() OVER (
            PARTITION BY month
            ORDER BY revenue DESC
        ) AS revenue_rank
    FROM monthly_revenue
)

SELECT
    month,
    PULocationID,
    revenue,
    revenue_rank
FROM locations
WHERE revenue_rank <= 5
ORDER BY month, revenue_rank
""")

monthly_ranking.show(5, truncate=False)

[Stage 106:==========================================>             (9 + 3) / 12]

+-----+------------+-------------------+------------+
|month|PULocationID|revenue            |revenue_rank|
+-----+------------+-------------------+------------+
|1    |132         |1.107120876000486E7|1           |
|1    |138         |5745056.440000303  |2           |
|1    |161         |3234330.899999876  |3           |
|1    |237         |2672667.109999881  |4           |
|1    |230         |2657710.9999998803 |5           |
+-----+------------+-------------------+------------+
only showing top 5 rows


In [47]:
revenue_quartiles = spark.sql("""
WITH monthly_revenue AS (
    SELECT
        month,
        PULocationID,
        SUM(total_amount) AS revenue
    FROM trips
    GROUP BY month, PULocationID
),

quartiles AS (
    SELECT
        month,
        PULocationID,
        revenue,
        NTILE(4) OVER (
            PARTITION BY month
            ORDER BY revenue DESC
        ) AS revenue_quartile
    FROM monthly_revenue
)

SELECT
    month,
    COUNT(*) AS top_quartile_locations
FROM quartiles
WHERE revenue_quartile = 1
GROUP BY month
ORDER BY month
""")

revenue_quartiles.show()

[Stage 112:=====================================>                  (8 + 4) / 12]

+-----+----------------------+
|month|top_quartile_locations|
+-----+----------------------+
|    1|                    64|
|    2|                    63|
|    3|                    63|
|    4|                    63|
|    5|                    63|
|    6|                    63|
|    7|                    64|
|    8|                    63|
|    9|                    63|
|   10|                    65|
|   11|                    64|
|   12|                    63|
+-----+----------------------+



In [14]:
# 3.4
daily_revenue = spark.sql("""
WITH daily_stats AS (
    SELECT
        DATE(tpep_pickup_datetime) AS trip_date,
        SUM(total_amount) AS daily_revenue
    FROM trips
    GROUP BY DATE(tpep_pickup_datetime)
)

SELECT
    trip_date,
    daily_revenue,
    SUM(daily_revenue) OVER (
        ORDER BY trip_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total_revenue,
    AVG(daily_revenue) OVER (
        ORDER BY trip_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS moving_avg_7d,
    daily_revenue - LAG(daily_revenue) OVER (
        ORDER BY trip_date
    ) AS revenue_change
FROM daily_stats
ORDER BY trip_date
""")

daily_revenue.show(10, truncate=False)

26/05/10 21:22:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 21:22:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 21:22:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 21:22:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 21:22:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 21:22:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 2

+----------+------------------+---------------------+------------------+-------------------+
|trip_date |daily_revenue     |running_total_revenue|moving_avg_7d     |revenue_change     |
+----------+------------------+---------------------+------------------+-------------------+
|2024-01-01|2102786.3999998653|2102786.3999998653   |2102786.3999998653|NULL               |
|2024-01-02|2179962.0899998774|4282748.489999743    |2141374.2449998716|77175.69000001205  |
|2024-01-03|2265092.2499999288|6547840.739999672    |2182613.5799998906|85130.16000005137  |
|2024-01-04|2687180.779999942 |9235021.519999614    |2308755.3799999035|422088.5300000133  |
|2024-01-05|2600641.5899999337|1.1835663109999549E7 |2367132.6219999096|-86539.19000000833 |
|2024-01-06|2279012.1899999455|1.4114675299999494E7 |2352445.883333249 |-321629.39999998827|
|2024-01-07|1799281.6599998763|1.5913956959999371E7 |2273422.422857053 |-479730.5300000692 |
|2024-01-08|2123706.3899999345|1.8037663349999305E7 |2276410.992857062

In [49]:
# Best selling day revenue
best_selling = spark.sql("""
WITH daily_stats AS (
    SELECT
        DATE(tpep_pickup_datetime) AS trip_date,
        SUM(total_amount) AS daily_revenue
    FROM trips
    GROUP BY DATE(tpep_pickup_datetime)
),

revenue_changes AS (
    SELECT
        trip_date,
        daily_revenue,
        daily_revenue - LAG(daily_revenue) OVER (
            ORDER BY trip_date
        ) AS revenue_change
    FROM daily_stats
)

SELECT *
FROM revenue_changes
ORDER BY revenue_change ASC
LIMIT 1
""")

best_selling.show()

26/05/10 18:13:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 18:13:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 18:13:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
[Stage 124:=====================================>                  (8 + 4) / 12]

+----------+------------------+--------------+
| trip_date|     daily_revenue|revenue_change|
+----------+------------------+--------------+
|2024-01-01|2102786.3999998653|          NULL|
+----------+------------------+--------------+



26/05/10 18:13:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 18:13:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 18:13:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 18:13:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
                                                                                

In [50]:
hourly_comparison = spark.sql("""
SELECT
    tpep_pickup_datetime,
    hour_of_day,
    total_amount,
    AVG(total_amount) OVER (
        PARTITION BY hour_of_day
    ) AS hourly_avg_total,
    total_amount - AVG(total_amount) OVER (
        PARTITION BY hour_of_day
    ) AS difference_hour
FROM trips
ORDER BY difference_hour DESC
""")

hourly_comparison.show()

[Stage 132:================================================>        (6 + 1) / 7]

+--------------------+-----------+------------+------------------+------------------+
|tpep_pickup_datetime|hour_of_day|total_amount|  hourly_avg_total|   difference_hour|
+--------------------+-----------+------------+------------------+------------------+
| 2024-12-04 14:24:18|         14|      3037.1| 30.10276034924424|3006.9972396507555|
| 2024-07-22 20:41:53|         20|     2265.45|28.448740387291007| 2237.001259612709|
| 2024-01-14 10:08:11|         10|      2225.3|27.318412750134847|2197.9815872498652|
| 2024-04-06 16:33:28|         16|     1737.18| 32.07531533721916| 1705.104684662781|
| 2024-11-06 09:05:55|          9|     1477.68|26.796420912531172| 1450.883579087469|
| 2024-06-30 18:20:04|         18|     1315.97| 28.68883199424098|1287.2811680057591|
| 2024-12-16 17:06:06|         17|     1218.99|30.211947044066807|1188.7780529559332|
| 2024-08-02 18:01:02|         18|     1204.41| 28.68883199424098|1175.7211680057592|
| 2024-08-06 23:14:08|         23|     1199.67|30.7480

In [51]:
highest_lowest_revenue = spark.sql("""
WITH daily_revenue AS (
    SELECT
        month,
        DATE(tpep_pickup_datetime) AS trip_date,
        SUM(total_amount) AS daily_revenue
    FROM trips
    GROUP BY month, DATE(tpep_pickup_datetime)
)

SELECT DISTINCT
    month,
    FIRST_VALUE(trip_date) OVER (
        PARTITION BY month
        ORDER BY daily_revenue DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS highest_revenue_day,
    FIRST_VALUE(daily_revenue) OVER (
        PARTITION BY month
        ORDER BY daily_revenue DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS highest_revenue,
    LAST_VALUE(trip_date) OVER (
        PARTITION BY month
        ORDER BY daily_revenue DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS lowest_revenue_day,
    LAST_VALUE(daily_revenue) OVER (
        PARTITION BY month
        ORDER BY daily_revenue DESC
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS lowest_revenue
FROM daily_revenue
ORDER BY month
""")

highest_lowest_revenue.show()

[Stage 133:==========================================>             (9 + 3) / 12]

+-----+-------------------+------------------+------------------+------------------+
|month|highest_revenue_day|   highest_revenue|lowest_revenue_day|    lowest_revenue|
+-----+-------------------+------------------+------------------+------------------+
|    1|         2024-01-25|2827199.5599999046|        2024-01-07|1799281.6599998763|
|    2|         2024-02-29|3175067.0899998588|        2024-02-13|1636503.4299999436|
|    3|         2024-03-14|  3355273.75999981|        2024-03-31|2303892.2699998557|
|    4|         2024-04-18| 3358428.489999808|        2024-04-22|2420538.5399998706|
|    5|         2024-05-16|3645496.9999997504|        2024-05-27|1895569.0399998894|
|    6|         2024-06-06|3584550.7399997856|        2024-06-30|2208535.3999998807|
|    7|         2024-07-18|3162952.4099998423|        2024-07-04|1489733.1299999114|
|    8|         2024-08-01|2959176.5499998326|        2024-08-18|2103687.6499998774|
|    9|         2024-09-26|3599853.9599997164|        2024-09-01|

In [52]:
total_revenue = spark.sql("""
SELECT
    month,
    payment_type,
    SUM(total_amount) AS total_revenue
FROM trips
GROUP BY ROLLUP(month, payment_type)
ORDER BY month, payment_type
""")

total_revenue.show()

[Stage 139:=====================================>                  (8 + 4) / 12]

+-----+------------+--------------------+
|month|payment_type|       total_revenue|
+-----+------------+--------------------+
| NULL|        NULL|1.0335206038685542E9|
|    1|        NULL| 7.461821098993786E7|
|    1|           1|6.3853269359997004E7|
|    1|           2|   9979769.850000516|
|    1|           3|  217962.16000000213|
|    1|           4|    567209.620000004|
|    2|        NULL| 7.444384533994398E7|
|    2|           1| 6.434241821999472E7|
|    2|           2|   9293911.060000887|
|    2|           3|  221516.05000000112|
|    2|           4|    586000.009999999|
|    3|        NULL| 8.601527200986908E7|
|    3|           1|  7.40164440499379E7|
|    3|           2|1.0960267689999329E7|
|    3|           3|   282517.2400000034|
|    3|           4|   756043.0299999764|
|    4|        NULL| 8.579874491987401E7|
|    4|           1| 7.367574222994539E7|
|    4|           2|1.1093835020000206E7|
|    4|           3|   274852.4400000013|
+-----+------------+--------------

# Part 4: Performance Analysis

In [15]:
# 4.1
daily_revenue.explain(True)

== Parsed Logical Plan ==
CTE [daily_stats]
:  +- 'SubqueryAlias daily_stats
:     +- 'Aggregate ['DATE('tpep_pickup_datetime)], ['DATE('tpep_pickup_datetime) AS trip_date#282, 'SUM('total_amount) AS daily_revenue#283]
:        +- 'UnresolvedRelation [trips], [], false
+- 'Sort ['trip_date ASC NULLS FIRST], true
   +- 'Project ['trip_date, 'daily_revenue, 'SUM('daily_revenue) windowspecdefinition('trip_date ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS running_total_revenue#279, 'AVG('daily_revenue) windowspecdefinition('trip_date ASC NULLS FIRST, specifiedwindowframe(RowFrame, -6, currentrow$())) AS moving_avg_7d#280, ('daily_revenue - 'LAG('daily_revenue) windowspecdefinition('trip_date ASC NULLS FIRST, unspecifiedframe$())) AS revenue_change#281]
      +- 'UnresolvedRelation [daily_stats], [], false

== Analyzed Logical Plan ==
trip_date: date, daily_revenue: double, running_total_revenue: double, moving_avg_7d: double, revenue_change: dou

26/05/10 21:22:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/10 21:22:45 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [16]:
import time
import builtins
# 4.2
query = """
WITH daily_stats AS (
    SELECT
        DATE(tpep_pickup_datetime) AS trip_date,
        SUM(total_amount) AS daily_revenue
    FROM trips
    GROUP BY DATE(tpep_pickup_datetime)
)

SELECT
    trip_date,
    daily_revenue,
    SUM(daily_revenue) OVER (
        ORDER BY trip_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total_revenue,
    AVG(daily_revenue) OVER (
        ORDER BY trip_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS moving_avg_7d,
    daily_revenue - LAG(daily_revenue) OVER (
        ORDER BY trip_date
    ) AS revenue_change
FROM daily_stats
ORDER BY trip_date
"""

times_no_cache = []

for i in range(3):
    start = time.time()
    spark.sql(query).count()
    end = time.time()
    times_no_cache.append(end - start)

avg_no_cache = builtins.sum(times_no_cache) / len(times_no_cache)
print("Average time without cache:", avg_no_cache)

[Stage 53:======================================>                  (8 + 4) / 12]

Average time without cache: 2.581538120905558


In [17]:
# Cache DataFrame
df_clean.cache()
df_clean.count()

times_cache = []
for i in range(3):
    start = time.time()
    spark.sql(query).count()
    end = time.time()
    times_cache.append(end - start)
avg_cache = builtins.sum(times_cache) / len(times_cache)

print("Average time with cache:", avg_cache)

26/05/10 21:23:11 WARN MemoryStore: Not enough space to cache rdd_131_7 in memory! (computed 214.4 MiB so far)
26/05/10 21:23:11 WARN BlockManager: Persisting block rdd_131_7 to disk instead.
26/05/10 21:23:11 WARN MemoryStore: Not enough space to cache rdd_131_4 in memory! (computed 213.8 MiB so far)
26/05/10 21:23:11 WARN BlockManager: Persisting block rdd_131_4 to disk instead.
26/05/10 21:23:11 WARN MemoryStore: Not enough space to cache rdd_131_6 in memory! (computed 214.5 MiB so far)
26/05/10 21:23:11 WARN BlockManager: Persisting block rdd_131_6 to disk instead.
26/05/10 21:23:23 WARN MemoryStore: Not enough space to cache rdd_131_7 in memory! (computed 40.0 MiB so far)
26/05/10 21:23:25 WARN MemoryStore: Not enough space to cache rdd_131_6 in memory! (computed 214.5 MiB so far)
26/05/10 21:23:26 WARN MemoryStore: Not enough space to cache rdd_131_4 in memory! (computed 136.7 MiB so far)
26/05/10 21:23:27 WARN MemoryStore: Not enough space to cache rdd_131_10 in memory! (compute

Average time with cache: 3.94199275970459


In [18]:
# 4.3
# Write partitioned dataset
df_clean.write.mode("overwrite").partitionBy("month").parquet("/app/data/nyc_taxi_partitioned")

26/05/10 21:26:31 WARN MemoryStore: Not enough space to cache rdd_131_7 in memory! (computed 78.7 MiB so far)
26/05/10 21:26:32 WARN MemoryStore: Not enough space to cache rdd_131_4 in memory! (computed 136.7 MiB so far)
26/05/10 21:26:32 WARN MemoryStore: Not enough space to cache rdd_131_5 in memory! (computed 137.0 MiB so far)
26/05/10 21:26:32 WARN MemoryStore: Not enough space to cache rdd_131_0 in memory! (computed 214.1 MiB so far)
26/05/10 21:27:09 WARN MemoryStore: Not enough space to cache rdd_131_10 in memory! (computed 214.7 MiB so far)
26/05/10 21:27:13 WARN MemoryStore: Not enough space to cache rdd_131_11 in memory! (computed 330.6 MiB so far)
                                                                                

In [ ]:
start = time.time()
single_file_df = spark.read.parquet("/app/data/nyc_taxi_2024_clean.parquet")
single_file_df.filter(col("month") == 1).count()
end = time.time()
print("Single file read time:", end - start)

In [ ]:
start = time.time()
partitioned_df = spark.read.parquet("/app/data/nyc_taxi_partitioned")
partitioned_df.filter(col("month") == 1).count()
end = time.time()
print("Partitioned read time:", end - start)

In [ ]:
partitioned_df.filter(col("month") == 1).explain(True)